In [16]:
# data_generation.py
import pandas as pd
import numpy as np

np.random.seed(42)

n_users = 1000
n_taskers = 200

users = pd.DataFrame({
    "user_id": range(n_users),
    "user_lat": np.random.uniform(49.0, 49.5, n_users),
    "user_lon": np.random.uniform(-123.5, -122.5, n_users),
})

taskers = pd.DataFrame({
    "tasker_id": range(n_taskers),
    "rating": np.random.uniform(3, 5, n_taskers),
    "price": np.random.uniform(20, 100, n_taskers),
    "completed_tasks": np.random.randint(10, 500, n_taskers),
    "lat": np.random.uniform(49.0, 49.5, n_taskers),
    "lon": np.random.uniform(-123.5, -122.5, n_taskers),
    
})

rows = []

for _ in range(10000):
    u = users.sample(1).iloc[0]
    t = taskers.sample(1).iloc[0]

    distance = np.sqrt((u.user_lat - t.lat)**2 + (u.user_lon - t.lon)**2)

    prob = 0.6*(t.rating/5) - 0.3*(distance) - 0.2*(t.price/100)
    prob = 1 / (1 + np.exp(-prob))

    clicked = np.random.binomial(1, prob)
    booked = np.random.binomial(1, prob * 0.7)

    rows.append([
        u.user_id, t.tasker_id, t.rating, t.price,
        t.completed_tasks, distance, clicked, booked
    ])

df = pd.DataFrame(rows, columns=[
    "user_id","tasker_id","rating","price",
    "completed_tasks","distance","clicked","booked"
])

df.to_csv("data.csv", index=False)
print("Data saved!")

Data saved!


In [18]:
# train.py
import pandas as pd
from lightgbm import LGBMRanker
from sklearn.model_selection import train_test_split

df = pd.read_csv("data.csv")

features = ["rating","price","completed_tasks","distance"]
target = "booked"

X = df[features]
y = df[target]

# group = تعداد آیتم‌ها برای هر user
group = df.groupby("user_id").size().to_list()

model = LGBMRanker(objective="lambdarank")

model.fit(
    X, y,
    group=group,
)
import joblib
joblib.dump(model, "model.pkl")

print("Model trained!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000491 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 821
[LightGBM] [Info] Number of data points in the train set: 10000, number of used features: 4
Model trained!


In [20]:
# predict.py
import pandas as pd
import joblib

model = joblib.load("model.pkl")

def rank_taskers(input_df):
    features = ["rating","price","completed_tasks","distance"]
    scores = model.predict(input_df[features])
    input_df["score"] = scores
    return input_df.sort_values(by="score", ascending=False)

# تست
df = pd.read_csv("data.csv").head(20)
ranked = rank_taskers(df)

print(ranked[["tasker_id","score"]].head())

    tasker_id     score
18      126.0  1.092876
4       103.0  0.747137
9        32.0  0.638473
15      135.0  0.426332
0       109.0  0.357513


In [ ]:
# from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
from sklearn.metrics import ndcg_score

app = FastAPI()

# load trained model
model = joblib.load("model.pkl")


# -----------------------------
# Data Model
# -----------------------------
class Tasker(BaseModel):
    rating: float
    price: float
    completed_tasks: int
    distance: float


# -----------------------------
# Ranking API
# -----------------------------
@app.post("/rank")
def rank(taskers: list[Tasker]):

    features = []

    for t in taskers:
        features.append([
            t.rating,
            t.price,
            t.completed_tasks,
            t.distance
        ])

    features = np.array(features)

    # ML prediction
    scores = model.predict(features)

    results = []

    for i, t in enumerate(taskers):

        base_score = scores[i]

        # fairness boost
        fairness = 0

        if t.completed_tasks < 50:
            fairness = 0.1

        final_score = base_score + fairness

        results.append({
            "tasker": t.dict(),
            "score": float(final_score),
            "base_score": float(base_score),
            "fairness_boost": fairness
        })

    # Sort ranking
    results = sorted(results, key=lambda x: x["score"], reverse=True)

    # -----------------------------
    # NDCG
    # -----------------------------
    try:

        y_true = np.array([[3, 2, 1][:len(scores)]])
        y_pred = np.array([scores[:len(y_true[0])]])

        ndcg = ndcg_score(y_true, y_pred)

    except Exception:
        ndcg = 0.0

    return {
        "ranking": results,
        "ndcg": float(ndcg)
    }

In [1]:
# ab_test.py
import pandas as pd

df = pd.read_csv("data.csv")

# baseline: فقط بر اساس rating
df["baseline_score"] = df["rating"]

# model: فرض کن score از مدل داری
df["model_score"] = df["booked"] + 0.1  # fake for demo

baseline_ctr = df.sort_values("baseline_score", ascending=False)["clicked"].mean()
model_ctr = df.sort_values("model_score", ascending=False)["clicked"].mean()

print("Baseline CTR:", baseline_ctr)
print("Model CTR:", model_ctr)

Baseline CTR: 0.5634
Model CTR: 0.5634
